In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

In [ ]:
dim_url_openaire = catalog.load('stg_openaire/dim_url_openaire')
dim_doi_openaire = catalog.load('stg_openaire/dim_doi_openaire')
dim_author_openalex = catalog.load('stg_openalex/dim_author_openalex')
dim_institution_openalex = catalog.load('stg_openalex/dim_institution_openalex')

bridge_publication_oaiidentifier_openaire = catalog.load('stg_openaire/bridge_publication_oaiidentifier_openaire')
bridge_publication_doi_openaire = catalog.load('stg_openaire/bridge_publication_doi_openaire')

bridge_author_institution_openalex = catalog.load('stg_openalex/bridge_author_institution_openalex')
bridge_publication_author_openalex = catalog.load('stg_openalex/bridge_publication_author_openalex')
bridge_publication_oai_id_openaire = catalog.load('stg_openaire/bridge_publication_oaiidentifier_openaire')

fact_publication_openalex = catalog.load('stg_openalex/fact_publication_openalex')
fact_publication_openaire = catalog.load('stg_openaire/fact_publication_openaire')
fact_publication_dspace = catalog.load('stg_dspace5/fact_publication_dspace5')


# 1. Estado actual del repositorio institucional (IR)


In [ ]:
fact_publication_dspace[['handle','title','type','dateavailable','dateissued']]

# 2. Superposición con OpenAIRE

En esta sección se identifican los ítems del repositorio que también están presentes en OpenAIRE, lo que permite recuperar métricas de uso como vistas y descargas.

Es posible filtrar las publicaciones recupero del IR **original_id**. En este caso se filtraran aquellos que contengan **oai:sedici.unlp.edu.ar:10915**, que es el prefijo de los identificador OAI que OpenAIRE recupera.


In [ ]:
sedici_filter = bridge_publication_oai_id_openaire["original_id"].str.contains("oai:sedici.unlp.edu.ar:10915", na=False)

dim_oai_id_openaire = bridge_publication_oai_id_openaire[sedici_filter].copy()

dim_oai_id_openaire["handle"] = dim_oai_id_openaire["original_id"].str.extract(r"(10915/\d+)")

dim_oai_id_openaire[['researchproduct_id','original_id','handle']]

Del código anterior podemos obtener el **researchproduct_id** de las publicaciones en OpenAIRE recuperadas del IR. También es posible deducir el **handle** de cada recurso en el IR a partir del **original_id**. Esto se hace porque OpenAIRE no modela el **handle**.

_There is an exception though: Handle(s) are minted by several repositories; as listing them all would not be a viable option, to avoid losing them as PIDs, Handles bypass the PID authority filtering rule. In all other cases, PIDs are included in the graph as alternate Identifiers._

Más info en https://graph.openaire.eu/docs/data-model/pids-and-identifiers/

# Publicaciones en OpenAIRE de IR con handle y DOI

In [ ]:
publication_ir_openaire = pd.merge(
    dim_oai_id_openaire,
    fact_publication_dspace,
    on='handle'
)

publication_ir_openaire[['researchproduct_id','handle','title','type','dateavailable','dateissued']]

# Merge con OpenAIRE 
a partir del _researchproduct_id_

In [ ]:
fact_publication_ir_openaire = pd.merge(
    fact_publication_openaire,
    publication_ir_openaire,
    on='researchproduct_id'
)

fact_publication_ir_openaire

# 3. Publicaciones en OpenAlex candidatas a ser depositadas

In [ ]:
publication_ir_openaire = pd.merge(
    publication_ir_openaire,
    bridge_publication_doi_openaire,
)

publication_ir_openaire = pd.merge(
    publication_ir_openaire,
    dim_doi_openaire
)

In [ ]:
publication_ir_openaire

In [ ]:
fact_publication = pd.merge(
    publication_ir_openaire,
    fact_publication_openalex,
    on='doi',
    how='outer'
)

In [ ]:
fact_publication

In [ ]:
not_in_ir = fact_publication['researchproduct_id'].isna()
publication_to_import = fact_publication[not_in_ir]
publication_to_import.drop(columns=['researchproduct_id','researchproduct_hk','handle','doi_hk','doi'], inplace=True)

In [ ]:
publication_to_import

# Autores de las publicaciones a importar

In [ ]:
ror_filter = dim_institution_openalex['ror'] == 'https://ror.org/01tjs6929'
dim_institution_openalex[ror_filter]

In [ ]:
institution_author_openalex = pd.merge(
    bridge_author_institution_openalex,
    dim_institution_openalex[ror_filter]
)

In [ ]:
institution_author_openalex[['author_institution_hk','author_hk']]

In [ ]:
institutional_authors = pd.merge(
    institution_author_openalex,
    dim_author_openalex,
    on='author_hk'
).sort_values(by=['works_count', 'cited_by_count'], ascending=False)

institutional_authors.drop(columns=['author_institution_hk','author_hk','institution_hk'], inplace=True)

In [ ]:
institutional_authors[['author_id','display_name_y','works_count','cited_by_count']]